# 02-03 Hive 基础

**JD 要求**: 熟悉 Hive/Spark 等大数据处理工具

本节用 DuckDB 模拟 Hive 的核心语法（HiveQL ≈ SQL，差异处会注明），覆盖大数据岗最常考的 Hive 知识点。

**本节目标**：
- 理解 Hive 的分区表、分桶表设计
- 掌握 ORC/Parquet 列存格式的优势
- 会写 Hive 特有语法：DISTRIBUTE BY、SORT BY、LATERAL VIEW
- 了解 Hive 查询优化（MapJoin、谓词下推、数据倾斜）

---

In [ ]:
import duckdb
import pandas as pd
import sys
sys.path.insert(0, "..")
from utils.data_generator import generate_ad_events

con = duckdb.connect()

# 加载数据
events = generate_ad_events(n=100000, seed=42)
events_df = pd.DataFrame(events)
events_df['timestamp'] = pd.to_datetime(events_df['timestamp'])
events_df['date'] = events_df['timestamp'].dt.date.astype(str)
events_df['hour'] = events_df['timestamp'].dt.hour
events_df['year'] = events_df['timestamp'].dt.year
events_df['month'] = events_df['timestamp'].dt.month
events_df['day'] = events_df['timestamp'].dt.day

con.execute("CREATE TABLE ad_events AS SELECT * FROM events_df")
print(f"数据加载: {len(events_df):,} 行")
print(f"列: {list(events_df.columns)}")

## 1. Hive 分区表（Partitioned Table）

Hive 分区 = HDFS 目录层级，按分区列过滤时直接跳过不相关目录（**分区裁剪**）

```sql
-- HiveQL 创建分区表
CREATE TABLE ad_events (
    event_id    BIGINT,
    user_id     STRING,
    ad_id       STRING,
    action      STRING,
    cost        DOUBLE
)
PARTITIONED BY (dt STRING, hour INT)  -- 分区列不在主表列中
STORED AS ORC;

-- 写入分区
INSERT INTO ad_events PARTITION(dt='2024-01-15', hour=10)
SELECT event_id, user_id, ad_id, action, cost
FROM staging_events
WHERE date = '2024-01-15' AND hour = 10;

-- 查询时只扫描需要的分区
SELECT * FROM ad_events WHERE dt = '2024-01-15';  -- 只读2024-01-15目录
```

**面试要点**：
- 分区过多（小文件问题）：每个分区下有大量小文件，NameNode 内存压力大
- 解决：合理选择分区粒度，通常按天或按天+小时

In [ ]:
# DuckDB 模拟分区查询（分区裁剪）
print("=== 模拟分区裁剪：只查某天某小时的数据 ===")

# 没有分区裁剪（全表扫描）
result_full = con.sql("""
SELECT date, hour, COUNT(*) AS events, SUM(cost) AS total_cost
FROM ad_events
WHERE date = '2024-01-15' AND hour = 10
GROUP BY date, hour
""").df()

# 有分区裁剪（等价于 Hive 中 WHERE dt='2024-01-15' AND hour=10）
result_part = con.sql("""
-- Hive 中这条查询只会读 /warehouse/ad_events/dt=2024-01-15/hour=10/ 目录
SELECT date, hour,
       COUNT(*) AS events,
       ROUND(SUM(cost), 2) AS total_cost,
       COUNT(*) FILTER (WHERE action = 'click') AS clicks
FROM ad_events
WHERE year = 2024 AND month = 1 AND day = 15 AND hour = 10
GROUP BY date, hour
""").df()

print(result_part.to_string())
print("\n在 Hive 中，分区裁剪可将扫描数据量从 100GB → 1GB（节省99%）")

## 2. 列存格式：ORC vs Parquet

| 特性 | ORC | Parquet |
|------|-----|---------|
| 生态 | Hive 原生，Hive 优化最好 | Spark/Presto/BigQuery 主流 |
| 压缩 | ZLIB/SNAPPY/ZSTD | SNAPPY/GZIP/ZSTD |
| 索引 | 内置轻量级索引 | 支持 Bloom Filter |
| 谓词下推 | 极好 | 极好 |
| 列裁剪 | ✅ 只读需要的列 | ✅ 只读需要的列 |

**为什么列存比行存快10-100x？**

```
行存（CSV/JSON）：
row1: [event_id, user_id, ad_id, action, cost, position, device, ...]
row2: [event_id, user_id, ad_id, action, cost, position, device, ...]

列存（ORC/Parquet）：
col_action:   [click, impression, click, convert, ...] ← 读 action 只读这一段
col_cost:     [0.5, 0.0, 0.3, 1.2, ...]              ← 相似值相邻，压缩率极高
col_user_id:  [u_123, u_456, ...]                     ← 不读 user_id 就完全跳过
```

In [ ]:
import os
import tempfile
import pyarrow as pa
import pyarrow.parquet as pq

# 演示列存格式的读写
arrow_table = pa.Table.from_pandas(events_df)

with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = os.path.join(tmpdir, 'ad_events.parquet')
    
    # 写 Parquet
    pq.write_table(arrow_table, parquet_path, compression='snappy')
    parquet_size = os.path.getsize(parquet_path)
    
    # 只读需要的列（列裁剪）
    df_selected = pq.read_table(parquet_path, columns=['action', 'cost', 'position']).to_pandas()
    
    print(f"原始 DataFrame 内存: {events_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    print(f"Parquet 文件大小: {parquet_size / 1024**2:.1f} MB")
    print(f"压缩率: {events_df.memory_usage(deep=True).sum() / parquet_size:.1f}x")
    print(f"\n列裁剪后只读 3 列，形状: {df_selected.shape}")
    print(df_selected.head(3))

## 3. Hive 特有语法

### 3.1 LATERAL VIEW EXPLODE（数组展开）

Hive 中常用于展开 JSON 数组字段，如广告标签、用户兴趣标签。

In [ ]:
# DuckDB 实现 EXPLODE 等价操作
# 模拟广告标签场景：每个广告有多个标签
ads_with_tags = pd.DataFrame({
    'ad_id': ['ad_1', 'ad_2', 'ad_3', 'ad_4'],
    'title': ['游戏皮肤', '美妆新品', '在线课程', '618大促'],
    'tags': [
        ['游戏', '皮肤', '动作'],
        ['美妆', '护肤', '女性'],
        ['教育', '编程', 'Python'],
        ['电商', '促销', '折扣'],
    ]
})
con.execute("CREATE TABLE ads_tags AS SELECT * FROM ads_with_tags")

print("=== LATERAL VIEW EXPLODE 等价（展开数组列）===")
print("""
-- HiveQL 写法：
SELECT ad_id, title, tag
FROM ads_tags
LATERAL VIEW EXPLODE(tags) t AS tag;
""")

# DuckDB 等价写法
con.sql("""
SELECT ad_id, title, UNNEST(tags) AS tag
FROM ads_tags
""").show()

In [ ]:
# DISTRIBUTE BY + SORT BY（Hive 特有，控制 Reducer 分配）
print("""
Hive 排序语法对比：

ORDER BY:      全局排序，只有 1 个 Reducer，大数据量慎用（OOM 风险）
SORT BY:       每个 Reducer 内部排序，多 Reducer 并行（局部有序）
DISTRIBUTE BY: 控制哪些 key 去哪个 Reducer（按 key 分发，不排序）
CLUSTER BY:    = DISTRIBUTE BY col + SORT BY col（分发且排序）

广告场景实例：
-- 每个广告主的数据发到同一个 Reducer，Reducer 内按日期排序
SELECT advertiser_id, date, SUM(cost) AS daily_cost
FROM ad_events
GROUP BY advertiser_id, date
DISTRIBUTE BY advertiser_id
SORT BY date;

-- 等价的全局排序（在大数据下很慢！）
SELECT advertiser_id, date, SUM(cost) AS daily_cost
FROM ad_events
GROUP BY advertiser_id, date
ORDER BY advertiser_id, date;  -- 单 Reducer，5TB 数据会超时
""")

# DuckDB 等价模拟
con.sql("""
SELECT advertiser_id, date,
       ROUND(SUM(cost), 2) AS daily_cost
FROM ad_events
GROUP BY advertiser_id, date
ORDER BY advertiser_id, date
LIMIT 8
""").show()

## 4. Hive 性能优化

### 4.1 MapJoin（小表 JOIN 大表）

In [ ]:
print("""
MapJoin 原理：
  小表（< 25MB）加载进每个 Mapper 的内存
  大表在 Map 阶段直接 JOIN，无需 Reduce 阶段
  性能提升：10x ~ 100x

-- HiveQL 写法 1: 自动 MapJoin（Hive 0.11+ 默认开启）
SET hive.auto.convert.join=true;
SET hive.mapjoin.smalltable.filesize=25000000;  -- 25MB 以内自动 MapJoin

SELECT e.*, a.title, a.ad_type
FROM ad_events e
JOIN ads a ON e.ad_id = a.ad_id;  -- ads 是小表，自动走 MapJoin

-- HiveQL 写法 2: 手动 hint（旧版本用）
SELECT /*+ MAPJOIN(a) */ e.*, a.title
FROM ad_events e
JOIN ads a ON e.ad_id = a.ad_id;
""")

# 模拟广告主维表 JOIN 事实表
advertisers_df = pd.DataFrame({
    'advertiser_id': [f'adv_{i}' for i in range(1, 6)],
    'name': ['游戏公司A', '美妆品牌B', '教育平台C', '电商平台D', '金融公司E'],
    'industry': ['游戏', '美妆', '教育', '电商', '金融']
})
con.execute("CREATE TABLE advertisers AS SELECT * FROM advertisers_df")

print("=== MapJoin 等价：维表 JOIN 事实表 ===")
con.sql("""
SELECT a.industry,
       COUNT(*) FILTER (WHERE e.action = 'click') AS clicks,
       ROUND(SUM(e.cost), 2) AS total_cost
FROM ad_events e
JOIN advertisers a ON e.advertiser_id = a.advertiser_id
GROUP BY a.industry
ORDER BY total_cost DESC
""").show()

In [ ]:
# 4.2 数据倾斜（Data Skew）处理
print("""
数据倾斜 = 某个 Reducer 处理的数据量远大于其他 Reducer

常见场景：
  - NULL 值多：JOIN 时 NULL key 全进一个 Reducer
  - 热点 key：广告 JOIN 时 adv_1（大广告主）数据量是 adv_5 的 1000 倍

解决方案 1：过滤 NULL
  SELECT *
  FROM events e
  JOIN ads a ON e.ad_id = a.ad_id
  WHERE e.ad_id IS NOT NULL;  -- 先过滤掉 NULL

解决方案 2：随机前缀打散（热点 key）
  -- 大表：给 key 加随机前缀 0-9
  SELECT CONCAT(CAST(FLOOR(RAND()*10) AS STRING), '_', ad_id) AS ad_id_salt,
         ...
  FROM large_table;
  
  -- 小表：复制 10 份，每份加不同前缀
  SELECT CONCAT(CAST(i AS STRING), '_', ad_id) AS ad_id_salt, ...
  FROM small_table
  LATERAL VIEW POSEXPLODE(ARRAY(0,1,2,3,4,5,6,7,8,9)) t AS i, v;

解决方案 3：开启 skewjoin
  SET hive.optimize.skewjoin=true;
  SET hive.skewjoin.key=100000;  -- 超过 10w 条的 key 认为是倾斜 key
""")

## 5. 广告数仓分层设计（面试常考）

大厂数仓一般分 ODS → DWD → DWS → ADS 四层：

In [ ]:
print("""
B站商业化数仓分层（典型设计）：

┌──────────────────────────────────────────────────────────────────┐
│  ADS 应用层 (Application Data Store)                              │
│  - ads_ad_daily_report   广告日报（给广告主看的）                  │
│  - ads_position_ctr      各广告位 CTR 汇总                        │
│  - ads_advertiser_roi    广告主 ROI 报表                          │
│  特点：直接对接 BI 报表/API，高度聚合，字段对业务友好              │
├──────────────────────────────────────────────────────────────────┤
│  DWS 汇总层 (Data Warehouse Service)                             │
│  - dws_ad_1d             广告日粒度汇总（clicks/impressions/cost）│
│  - dws_user_ad_7d        用户7日广告行为汇总                     │
│  - dws_advertiser_1d     广告主日消耗汇总                        │
│  特点：按主题按时间粒度聚合，DWD 的进一步加工                     │
├──────────────────────────────────────────────────────────────────┤
│  DWD 明细层 (Data Warehouse Detail)                              │
│  - dwd_ad_event_di       广告事件明细（清洗后，日期分区）         │
│  - dwd_user_info         用户基础信息（SCD2 拉链表）             │
│  特点：1:1 对应原始事件，清洗/标准化/去重，保留明细               │
├──────────────────────────────────────────────────────────────────┤
│  ODS 原始层 (Operational Data Store)                             │
│  - ods_ad_click_log      原始点击日志（Kafka → Hive）            │
│  - ods_ad_impression_log 原始曝光日志                           │
│  特点：原始数据，不做任何处理，保留原始字段，按天分区             │
└──────────────────────────────────────────────────────────────────┘

关键原则：
  - 数据只向上流动（ODS → DWD → DWS → ADS）
  - 下游依赖上游，上游变更需通知下游
  - ODS 保留原始数据，DWD 是唯一清洗层
  - ADS 层可以跨层直接用 DWD（避免 DWS 聚合损失明细）
""")

# 模拟从 ODS → DWD → DWS 的数据流
print("=== 模拟 DWD 层：清洗去重 ===")
con.sql("""
-- DWD 层：去掉重复事件，标准化字段
WITH dwd_ad_event AS (
    SELECT
        event_id,
        user_id,
        ad_id,
        advertiser_id,
        action,
        LOWER(device) AS device,       -- 标准化设备名
        COALESCE(cost, 0.0) AS cost,   -- NULL 转 0
        date,
        hour,
        ROW_NUMBER() OVER (
            PARTITION BY event_id      -- 按 event_id 去重
            ORDER BY timestamp
        ) AS rn
    FROM ad_events
)
SELECT action, device, COUNT(*) AS event_cnt
FROM dwd_ad_event
WHERE rn = 1  -- 只保留第一条（去重）
GROUP BY action, device
ORDER BY event_cnt DESC
LIMIT 8
""").show()

print("=== 模拟 DWS 层：日粒度聚合 ===")
con.sql("""
SELECT
    date,
    COUNT(*) FILTER (WHERE action = 'impression') AS impressions,
    COUNT(*) FILTER (WHERE action = 'click')      AS clicks,
    COUNT(*) FILTER (WHERE action = 'convert')    AS converts,
    ROUND(SUM(cost), 2)                           AS total_cost,
    ROUND(100.0 * COUNT(*) FILTER (WHERE action = 'click') /
          NULLIF(COUNT(*) FILTER (WHERE action = 'impression'), 0), 2) AS ctr_pct
FROM ad_events
GROUP BY date
ORDER BY date
LIMIT 7
""").show()

## 6. Hive 面试速查

| 问题 | 要点 |
|------|------|
| Hive vs MySQL | Hive 面向 OLAP 批处理，不支持事务，延迟秒级到小时；MySQL 面向 OLTP，毫秒级 |
| ORC vs Parquet | ORC 是 Hive 原生最优；Parquet 是 Spark/跨系统主流 |
| ORDER BY vs SORT BY | ORDER BY 全局排序单 Reducer；SORT BY 每个 Reducer 内排序 |
| MapJoin 条件 | 小表 < 25MB（可调），加载进 Map 内存，避免 Shuffle |
| 数据倾斜解决 | 过滤 NULL + 随机前缀打散 + skewjoin 参数 |
| 小文件问题 | 合并小文件：`SET hive.merge.mapfiles=true`；或用 INSERT OVERWRITE 重写分区 |

**自检**: 能说清楚 ODS/DWD/DWS/ADS 各层的职责和数据流向吗？

**下一节**: `../03-llm-fundamentals/02_tokenization_embedding.ipynb`